# UTS 2 — Eksperimen Arsitektur 3: ResNet-50 (224×224)
**Proyek UTS 2 — Klasifikasi 4 Penyakit Daun Padi (*Bacterialblight, Blast, Brownspot, Tungro*)**  
**Tim:** Nasarius Viatrus Bili (`235314181`) & Yohanes Paskalis Rafllyno (`245314038`)  
**Arsitektur:** ResNet-50 Pretrained ImageNet (~23,5 juta parameter, 50 layer *bottleneck residual* + GAP)  
**Environment:** Kaggle Notebook (Aktifkan **Settings → Accelerator: GPU T4 x2** & **Internet: ON** — otomatis menjalankan `nn.DataParallel` + Mixed Precision FP16)

### Daftar Isi Pipeline Lengkap (Sesuai Checklist Presentasi Proyek UTS 2)
1. **Setup & Unduh Dataset Bersih** (`gdown` + `unzip` + Multi-GPU `T4 x2` `DataParallel`)
2. **Show Dataset (EDA & Bukti Data Cleaning)** — audit 5.932 → 4.752 citra unik, galeri 4×4 sampel, profil warna RGB
3. **Split Dataset** — verifikasi *Zero-Leakage Group-Aware Stratified Split* (`70% Train : 15% Val : 15% Test`) & grafik proporsi
4. **Preprocessing & Augmentasi / Ekstraksi Fitur** — resize seragam $224\times224$, normalisasi ImageNet, visualisasi *Basic* vs *Aggressive Augmentation*
5. **Load Model** — definisi arsitektur, summary parameter, & *dummy forward pass*
6. **Training Model (Visualisasi) & Validasi** — Exp 01, Exp 02, Exp 03 + **kurva Train vs Val (+ garis Test) untuk Loss & Accuracy pada masing-masing eksperimen** guna mendiagnosa *overfitting*
7. **Testing (Visualisasi)** — galeri prediksi benar vs *error analysis* (prediksi salah), histogram *confidence*, dan **Grad-CAM**
8. **Evaluasi Model** — tabel rekap ablation, *Confusion Matrix* (jumlah & persentase), metrik Precision/Recall/F1 per kelas, **Kurva ROC-AUC Multi-Kelas**, dan ekspor hasil JSON

### Rancangan 3 Eksperimen Terkontrol (*Ablation Study*)
* **Exp 01 — Feature Extraction (Maks. 10 Epoch, EarlyStopping patience=4):** *Backbone* dibekukan penuh; hanya *classifier head* yang dilatih dengan augmentasi standar (`train_loader_basic`).
* **Exp 02 — Progressive Fine-Tuning (2 Tahap × 8 Epoch, EarlyStopping patience=3 per tahap):** Tahap 1 melatih *head* (8 epoch), dilanjutkan Tahap 2 membuka blok konvolusi atas (8 epoch) dengan *differential LR*. **Augmentasi & DataLoader identik dengan Exp 01** agar peningkatan performa terisolasi murni dari *fine-tuning*.
* **Exp 03 — Advanced Generalization / Final Optimization (Maks. 20 Epoch, EarlyStopping patience=6):** Memulai dari bobot *head* terlatih (mencegah *catastrophic forgetting*), lalu **unfreeze seluruh layer** dengan **3-tier differential LR**, augmentasi agresif terukur (`RandomResizedCrop`, `RandomAffine`, `ColorJitter` dengan `hue<=0.02`, `RandomErasing`), `label_smoothing=0.08`, *weight decay* `1e-2`, dan *gradient clipping*.

## 1. Setup Lingkungan & Unduh Dataset Bersih (GDrive via `gdown`)

In [ ]:
!pip install -q gdown

import os
import gc
import json
import time
import copy
import random
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    roc_curve,
    auc,
)
from sklearn.preprocessing import label_binarize

# Reproducibility & Konfigurasi Multi-GPU (Kaggle T4 x2)
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True  # Optimasi Tensor Core T4 untuk input tetap 224x224

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()
USE_MULTI_GPU = NUM_GPUS > 1

print(f"Device aktif: {DEVICE} | Jumlah GPU terdeteksi: {NUM_GPUS}")
for i in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"  - GPU {i}: {props.name} ({props.total_memory / 1e9:.1f} GB VRAM)")
if USE_MULTI_GPU:
    print("Mode Multi-GPU AKTIF: menggunakan nn.DataParallel + Mixed Precision (FP16) pada T4 x2!")

In [ ]:
# ==============================================================================
# KONFIGURASI UNDUH DATASET BERSIH DARI GOOGLE DRIVE
# ==============================================================================
# Cara mendapat ID: Klik kanan Rice_Leaf_Clean_Split.zip di Google Drive
# -> Share (Anyone with the link) -> Salin FILE_ID dari URL:
# https://drive.google.com/file/d/<FILE_ID>/view?usp=sharing
GDRIVE_FILE_ID = "GANTI_DENGAN_FILE_ID_GDRIVE_ANDA"

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working")
WORK_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = WORK_DIR / "Rice_Leaf_Clean_Split.zip"
EXTRACT_DIR = WORK_DIR / "dataset_clean"

assert GDRIVE_FILE_ID != "GANTI_DENGAN_FILE_ID_GDRIVE_ANDA", (
    "Silakan ganti GDRIVE_FILE_ID terlebih dahulu dengan ID file Google Drive Anda!"
)

if not EXTRACT_DIR.exists() or not any(EXTRACT_DIR.iterdir()):
    import gdown
    gdown.download(id=GDRIVE_FILE_ID, output=str(ZIP_PATH), quiet=False)
    assert ZIP_PATH.exists(), "Download ZIP gagal! Pastikan akses file Drive = Anyone with the link."
    !unzip -q -o "{ZIP_PATH}" -d "{EXTRACT_DIR}"
    print("Ekstraksi selesai.")

def find_split_root(base_dir: Path) -> Path:
    if (base_dir / "train").is_dir() and (base_dir / "val").is_dir() and (base_dir / "test").is_dir():
        return base_dir
    for p in base_dir.rglob("train"):
        if p.is_dir() and (p.parent / "val").is_dir() and (p.parent / "test").is_dir():
            return p.parent
    raise FileNotFoundError(f"Struktur train/val/test tidak ditemukan di {base_dir}")

DATA_ROOT = find_split_root(EXTRACT_DIR)
print("DATA_ROOT terdeteksi:", DATA_ROOT)

## 2. Show Dataset (EDA, Bukti Data Cleaning, & Galeri Sampel)

In [ ]:
# ==============================================================================
# SHOW DATASET: EDA, BUKTI DATA CLEANING, GALERI 4x4 SAMPEL, & PROFIL WARNA
# ==============================================================================
# Tampilkan laporan audit cleaning lokal jika tersedia di dalam ZIP
report_candidates = list(EXTRACT_DIR.rglob("cleaning_report.json"))
if report_candidates:
    clean_rep = json.loads(report_candidates[0].read_text(encoding="utf-8"))
    print("=== LAPORAN AUDIT & DATA CLEANING LOKAL ===")
    print(f"  Citra mentah awal         : {clean_rep.get('raw_total', 5932):,}")
    print(f"  Duplikat dibuang          : {clean_rep.get('total_duplicates_removed', 1180):,} "
          f"(MD5: {clean_rep.get('raw_md5_duplicates_removed', 1138)}, Pixel-identik: {clean_rep.get('extra_pixel_duplicates_removed', 42)})")
    print(f"  Citra unik bersih final   : {clean_rep.get('clean_unique_total', 4752):,}")
    print(f"  Resolusi terstandardisasi : {clean_rep.get('target_size', [300, 300])} RGB (Square Center Crop untuk Tungro)")

classes = sorted([d for d in os.listdir(DATA_ROOT / "train") if (DATA_ROOT / "train" / d).is_dir()])

# 1. Galeri Visual 4 Kelas x 4 Sampel (Show Dataset langsung di notebook)
fig, axes = plt.subplots(len(classes), 4, figsize=(12, 3.0 * len(classes)))
for r, cls in enumerate(classes):
    sample_fns = sorted(os.listdir(DATA_ROOT / "train" / cls))[:4]
    for c, fn in enumerate(sample_fns):
        im = Image.open(DATA_ROOT / "train" / cls / fn)
        axes[r, c].imshow(im)
        if c == 0:
            axes[r, c].set_ylabel(cls, fontsize=11, fontweight="bold")
        axes[r, c].set_title(f"{cls}\n{fn[:18]} ({im.size[0]}x{im.size[1]})", fontsize=8.5)
        axes[r, c].set_xticks([])
        axes[r, c].set_yticks([])
plt.suptitle("Show Dataset — Galeri 16 Sampel Citra Daun Padi (4 Kelas × 4 Contoh)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(WORK_DIR / "01_show_dataset_gallery.png", dpi=150)
plt.show()

# 2. Profil Intensitas Warna RGB Rata-rata per Kelas (EDA Karakteristik Visual)
mean_rgb = {}
for cls in classes:
    sample_files = sorted(os.listdir(DATA_ROOT / "train" / cls))[:120]
    vals = [np.asarray(Image.open(DATA_ROOT / "train" / cls / fn), dtype=np.float32).mean(axis=(0, 1)) for fn in sample_files]
    mean_rgb[cls] = np.mean(vals, axis=0)
df_rgb = pd.DataFrame(mean_rgb, index=["Red (R)", "Green (G)", "Blue (B)"]).T

fig, ax = plt.subplots(figsize=(8.5, 4.2))
df_rgb.plot(kind="bar", ax=ax, color=["#EF4444", "#22C55E", "#3B82F6"])
ax.set_title("EDA — Profil Intensitas Warna RGB Rata-rata per Kelas Penyakit Daun Padi")
ax.set_ylabel("Intensitas Piksel Rata-rata (0 - 255)")
ax.tick_params(axis="x", rotation=15)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Split Dataset (Verifikasi Zero-Leakage Stratified Group Split 70:15:15)

In [ ]:
# ==============================================================================
# SPLIT DATASET: VERIFIKASI ZERO-LEAKAGE GROUP-AWARE STRATIFIED SPLIT (70:15:15)
# ==============================================================================
splits = ["train", "val", "test"]
split_counts = {}
md5_sets = {}
non_std = []

for sp in splits:
    sp_dir = DATA_ROOT / sp
    cls_counts = {}
    md5_sets[sp] = set()
    for cls in sorted(os.listdir(sp_dir)):
        cdir = sp_dir / cls
        if not cdir.is_dir():
            continue
        files = sorted(os.listdir(cdir))
        cls_counts[cls] = len(files)
        for fn in files:
            fp = cdir / fn
            with open(fp, "rb") as f:
                md5_sets[sp].add(hashlib.md5(f.read()).hexdigest())
            if len(non_std) < 10:
                with Image.open(fp) as im:
                    if im.size != (300, 300) or im.mode != "RGB":
                        non_std.append((str(fp), im.size, im.mode))
    split_counts[sp] = cls_counts

df_counts = pd.DataFrame(split_counts)
df_counts["Total"] = df_counts.sum(axis=1)
df_counts.loc["TOTAL"] = df_counts.sum(axis=0)
print("=== RINGKASAN JUMLAH CITRA PER KELAS & SPLIT (70:15:15) ===")
display(df_counts)

print("\n=== AUDIT INTEGRITAS & ANTI-LEAKAGE ===")
print("Overlap MD5 train-val       :", len(md5_sets["train"] & md5_sets["val"]))
print("Overlap MD5 train-test      :", len(md5_sets["train"] & md5_sets["test"]))
print("Overlap MD5 val-test        :", len(md5_sets["val"] & md5_sets["test"]))
print("Sampel non-300x300 / non-RGB:", len(non_std))

# Visualisasi Proporsi Split (Pie Chart + Stacked Bar per Kelas)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
totals_sp = [df_counts.loc["TOTAL", sp] for sp in splits]
axes[0].pie(
    totals_sp,
    labels=[f"{sp.capitalize()}\n({n:,} | {n/sum(totals_sp)*100:.1f}%)" for sp, n in zip(splits, totals_sp)],
    colors=["#2563EB", "#F59E0B", "#10B981"],
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5},
)
axes[0].set_title("Proporsi Split Global (Zero-Leakage Group Split)")

df_plot = df_counts.drop(index="TOTAL")[splits]
df_plot.plot(kind="bar", stacked=True, ax=axes[1], color=["#2563EB", "#F59E0B", "#10B981"])
axes[1].set_title("Distribusi Split per Kelas Penyakit")
axes[1].set_ylabel("Jumlah Citra")
axes[1].tick_params(axis="x", rotation=15)
axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(WORK_DIR / "02_split_dataset_distribution.png", dpi=150)
plt.show()

## 4. Preprocessing & Augmentasi / Ekstraksi Data ($224 \times 224$)

In [ ]:
# ==============================================================================
# PREPROCESSING & AUGMENTASI (224x224, NORMALISASI IMAGENET, DATALOADER T4 x2)
# ==============================================================================
IMG_SIZE = 224
# Pada Kaggle T4 x2, batch_size=64 otomatis terbagi 32 citra per GPU T4 (optimal untuk Tensor Cores)
BATCH_SIZE = 64 if USE_MULTI_GPU else 32
NUM_WORKERS = min(4, os.cpu_count() or 2)
PERSISTENT_WORKERS = NUM_WORKERS > 0

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# 1. Augmentasi Standar (Dipakai identik pada Exp 01 & Exp 02 agar perbandingan terkontrol!)
basic_train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# 2. Augmentasi Agresif Terukur (Dipakai pada Exp 03: Advanced Generalization)
# Catatan CV Engineer:
# - RandomResizedCrop + RandomAffine + RandomErasing mensimulasikan variasi jarak foto,
#   sudut miring, dan oklusi daun/tangan di sawah.
# - Hue jitter dibatasi ketat (<= 0.02) agar ciri warna kuning-oranye Tungro tidak
#   berubah menjadi coklat seperti Brownspot!
aggressive_train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.72, 1.0), ratio=(0.85, 1.15)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=30),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), shear=8),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.20, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.12), ratio=(0.3, 3.3), value="random"),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds_basic = datasets.ImageFolder(DATA_ROOT / "train", transform=basic_train_transform)
train_ds_aggr  = datasets.ImageFolder(DATA_ROOT / "train", transform=aggressive_train_transform)
val_ds         = datasets.ImageFolder(DATA_ROOT / "val",   transform=eval_transform)
test_ds        = datasets.ImageFolder(DATA_ROOT / "test",  transform=eval_transform)

CLASS_NAMES = train_ds_basic.classes
NUM_CLASSES = len(CLASS_NAMES)

# drop_last=True pada train_loader mencegah sisa batch 1 sampel yang memicu error BatchNorm pada DataParallel
train_loader_basic = DataLoader(
    train_ds_basic, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True, persistent_workers=PERSISTENT_WORKERS
)
train_loader_aggr = DataLoader(
    train_ds_aggr, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True, persistent_workers=PERSISTENT_WORKERS
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=PERSISTENT_WORKERS
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=PERSISTENT_WORKERS
)

counts = np.bincount(train_ds_basic.targets)
class_weights = torch.tensor(counts.sum() / (NUM_CLASSES * counts), dtype=torch.float32, device=DEVICE)
print(f"Kelas ({NUM_CLASSES}): {CLASS_NAMES} | Batch Size: {BATCH_SIZE} ({'32/GPU x2' if USE_MULTI_GPU else 'single GPU'}) | Workers: {NUM_WORKERS}")
print(f"Train: {len(train_ds_basic)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
print("Class weights:", {c: round(float(w), 3) for c, w in zip(CLASS_NAMES, class_weights)})

# Helper denormalisasi untuk visualisasi tensor
def denorm(tensor):
    arr = tensor.cpu().numpy().transpose(1, 2, 0)
    arr = arr * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(arr, 0, 1)

# Visualisasi Hasil Preprocessing & Perbandingan Augmentasi Basic (Exp01-02) vs Aggressive (Exp03)
sample_path, sample_lbl = train_ds_basic.samples[0]
orig_pil = Image.open(sample_path).convert("RGB")
fig, axes = plt.subplots(2, 5, figsize=(14, 5.8))
axes[0, 0].imshow(orig_pil.resize((IMG_SIZE, IMG_SIZE)))
axes[0, 0].set_title(f"Asli (Resize 224)\n{CLASS_NAMES[sample_lbl]}", fontsize=9, fontweight="bold")
axes[1, 0].imshow(orig_pil.resize((IMG_SIZE, IMG_SIZE)))
axes[1, 0].set_title(f"Asli (Referensi)\n{CLASS_NAMES[sample_lbl]}", fontsize=9, fontweight="bold")

for k in range(1, 5):
    axes[0, k].imshow(denorm(basic_train_transform(orig_pil)))
    axes[0, k].set_title(f"Basic Aug #{k}\n(Exp 01 & 02)", fontsize=9)
    axes[1, k].imshow(denorm(aggressive_train_transform(orig_pil)))
    axes[1, k].set_title(f"Aggressive Aug #{k}\n(Exp 03 + Erasing)", fontsize=9)

for ax_row in axes:
    for a in ax_row:
        a.axis("off")
plt.suptitle("Visualisasi Preprocessing & Augmentasi Data (Basic vs Aggressive)", fontsize=12.5, fontweight="bold")
plt.tight_layout()
plt.savefig(WORK_DIR / "03_preprocessing_augmentation_preview.png", dpi=150)
plt.show()

## 5. Load Model & Engine Pelatihan (Multi-GPU T4 x2 + `tqdm` + EarlyStopping)

In [ ]:
MODEL_NAME = "ResNet-50 (Transfer Learning)"
MODEL_SLUG = "resnet50"

def build_model_for_exp(exp_key=None):
    m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    m.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(m.fc.in_features, NUM_CLASSES),
    )
    return m.to(DEVICE)

def get_gradcam_layer(model, exp_key=None):
    return model.layer4[-1]

def set_freeze_exp01(m):
    for name, p in m.named_parameters():
        p.requires_grad = name.startswith("fc.")
    return [{"params": m.fc.parameters(), "lr": 1e-3}]

def set_unfreeze_exp02_stage2(m):
    for name, p in m.named_parameters():
        p.requires_grad = any(name.startswith(k) for k in ("layer3.", "layer4.", "fc."))
    return [
        {"params": list(m.layer3.parameters()) + list(m.layer4.parameters()), "lr": 5e-5},
        {"params": m.fc.parameters(), "lr": 3e-4},
    ]

def set_full_unfreeze_exp03(m):
    for p in m.parameters(): p.requires_grad = True
    early = [p for name, p in m.named_parameters() if any(name.startswith(k) for k in ("conv1", "bn1", "layer1.", "layer2."))]
    deep  = [p for name, p in m.named_parameters() if any(name.startswith(k) for k in ("layer3.", "layer4."))]
    head  = list(m.fc.parameters())
    return [
        {"params": early, "lr": 1e-5},
        {"params": deep,  "lr": 5e-5},
        {"params": head,  "lr": 3e-4},
    ]

# Verifikasi Load Model & Dummy Forward Pass
_tmp = build_model_for_exp()
with torch.no_grad():
    _out = _tmp(torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE))
print(f"Load Model OK: {MODEL_NAME} | Total Params: {sum(p.numel() for p in _tmp.parameters()):,} | Output shape: {tuple(_out.shape)}")
print("Classifier Head:", _tmp.fc)
del _tmp, _out

In [ ]:
def unwrap_model(m):
    return m.module if isinstance(m, nn.DataParallel) else m


def wrap_multi_gpu(m):
    if USE_MULTI_GPU and not isinstance(m, nn.DataParallel):
        return nn.DataParallel(m)
    return m


def run_epoch(model, loader, criterion, optimizer=None, scaler=None, grad_clip=None, desc=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    total_samples = 0
    total_correct = 0
    all_preds, all_targets = [], []

    pbar = tqdm(loader, desc=desc or ("Train" if is_train else "Val"), leave=False, dynamic_ncols=True)
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for imgs, targets in pbar:
            imgs = imgs.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model(imgs)
                loss = criterion(logits, targets)

            if is_train:
                if scaler is not None:
                    scaler.scale(loss).backward()
                    if grad_clip is not None:
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    if grad_clip is not None:
                        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    optimizer.step()

            bs = imgs.size(0)
            total_loss += loss.item() * bs
            total_samples += bs
            preds = logits.argmax(dim=1)
            total_correct += (preds == targets).sum().item()
            all_preds.extend(preds.detach().cpu().numpy())
            all_targets.extend(targets.detach().cpu().numpy())

            pbar.set_postfix(
                loss=f"{total_loss / total_samples:.4f}",
                acc=f"{total_correct / total_samples:.4f}",
            )

    avg_loss = total_loss / max(total_samples, 1)
    acc = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average="macro")
    return avg_loss, acc, macro_f1


def get_early_stopping_patience(epochs: int) -> int:
    # Skala patience EarlyStopping yang proporsional terhadap total epoch:
    # 8 epoch -> 3 | 10 epoch -> 4 | 20 epoch -> 6 | 30 epoch -> 8 | 50 epoch -> 11
    if epochs <= 8:
        return 3
    if epochs <= 12:
        return 4
    return max(4, int(round(2.4 + 0.175 * epochs)))


def train_phase(model, train_loader, val_loader, criterion, optimizer, scheduler,
                epochs, history, patience=None, min_delta=1e-4, phase_name="Train",
                best_state=None, best_val_f1=-1.0, grad_clip=None):
    if patience is None:
        patience = get_early_stopping_patience(epochs)
    print(f"--> Memulai {phase_name} | max_epochs={epochs} | EarlyStopping patience={patience} (min_delta={min_delta})")

    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    no_improve = 0
    early_stopped = False

    epoch_bar = tqdm(range(1, epochs + 1), desc=f"{phase_name} Epochs", leave=True, dynamic_ncols=True)
    for ep in epoch_bar:
        t0 = time.time()
        tr_loss, tr_acc, tr_f1 = run_epoch(
            model, train_loader, criterion, optimizer, scaler,
            grad_clip=grad_clip, desc=f"[{phase_name}] Ep {ep:02d}/{epochs:02d} Train"
        )
        va_loss, va_acc, va_f1 = run_epoch(
            model, val_loader, criterion,
            desc=f"[{phase_name}] Ep {ep:02d}/{epochs:02d} Val"
        )
        if scheduler is not None:
            scheduler.step()
        dt = time.time() - t0

        history["phase"].append(phase_name)
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(va_acc)
        history["train_f1"].append(tr_f1)
        history["val_f1"].append(va_f1)

        improved = ""
        if va_f1 > best_val_f1 + min_delta:
            best_val_f1 = va_f1
            best_state = copy.deepcopy(unwrap_model(model).state_dict())
            no_improve = 0
            improved = " *"
        else:
            no_improve += 1

        epoch_bar.set_postfix(
            tr_loss=f"{tr_loss:.3f}",
            va_f1=f"{va_f1:.4f}",
            best=f"{best_val_f1:.4f}",
            es=f"{no_improve}/{patience}",
        )
        tqdm.write(
            f"[{phase_name}] Ep {ep:02d}/{epochs:02d} | "
            f"TrLoss={tr_loss:.4f} Acc={tr_acc:.4f} F1={tr_f1:.4f} | "
            f"VaLoss={va_loss:.4f} Acc={va_acc:.4f} F1={va_f1:.4f} ({dt:.1f}s){improved}"
        )

        if no_improve >= patience:
            early_stopped = True
            tqdm.write(f"[EarlyStopping] Berhenti normal di {phase_name} epoch {ep}/{epochs} (patience={patience} tercapai; bar <100% bukan error).")
            break

    history.setdefault("early_stopped_phases", []).append({
        "phase": phase_name,
        "max_epochs": epochs,
        "epochs_run": ep,
        "patience": patience,
        "early_stopped": early_stopped,
    })
    return best_state, best_val_f1


def evaluate_on_test(model, state_dict, loader, criterion):
    unwrap_model(model).load_state_dict(state_dict)
    model.eval()
    y_true, y_pred, y_probs = [], [], []
    latencies_ms = []
    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for imgs, targets in tqdm(loader, desc="Evaluasi Test Set", leave=False, dynamic_ncols=True):
            imgs = imgs.to(DEVICE, non_blocking=True)
            targets_dev = targets.to(DEVICE, non_blocking=True)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            logits = model(imgs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            dt_ms = (time.perf_counter() - t0) * 1000.0

            loss = criterion(logits, targets_dev)
            bs = imgs.size(0)
            total_loss += loss.item() * bs
            total_samples += bs
            probs = torch.softmax(logits, dim=1).cpu().numpy()

            latencies_ms.append(dt_ms / bs)
            y_probs.append(probs)
            y_pred.extend(probs.argmax(axis=1))
            y_true.extend(targets.numpy())

    return {
        "y_true": np.array(y_true),
        "y_pred": np.array(y_pred),
        "y_probs": np.vstack(y_probs),
        "test_loss": float(total_loss / max(total_samples, 1)),
        "test_acc": float(accuracy_score(y_true, y_pred)),
        "test_macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "inference_ms": float(np.mean(latencies_ms)),
    }


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

ALL_EXPERIMENTS = {}

## 6. Training Model & Validasi (Exp 01, Exp 02, Exp 03)

### Exp 01: Feature Extraction (Maks. 10 Epoch, EarlyStopping patience=4, Augmentasi Standar)

In [ ]:
# Exp 01: Feature Extraction (Backbone Beku, hanya melatih classifier head)
seed_everything(SEED)
EPOCHS_E1 = 10
PATIENCE_E1 = get_early_stopping_patience(EPOCHS_E1)  # 10 -> 4

base_e1 = build_model_for_exp("Exp01")
total_p1 = sum(p.numel() for p in base_e1.parameters())
param_groups_e1 = set_freeze_exp01(base_e1)
trainable_e1 = sum(p.numel() for p in base_e1.parameters() if p.requires_grad)
print(f"[Exp 01] Trainable params: {trainable_e1:,} / {total_p1:,}")
model_e1 = wrap_multi_gpu(base_e1)

criterion_e1 = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
opt_e1 = torch.optim.AdamW(param_groups_e1, weight_decay=1e-4)
sch_e1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_e1, T_max=EPOCHS_E1, eta_min=1e-5)

hist_e1 = {"phase": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "train_f1": [], "val_f1": []}
t0 = time.time()
best_s1, best_f1_1 = train_phase(
    model_e1, train_loader_basic, val_loader, criterion_e1, opt_e1, sch_e1,
    epochs=EPOCHS_E1, history=hist_e1, patience=PATIENCE_E1, phase_name="Exp01-FE"
)
dt1 = time.time() - t0
res_e1 = evaluate_on_test(model_e1, best_s1, test_loader, criterion_e1)
ALL_EXPERIMENTS["Exp01"] = {
    "desc": "Feature Extraction (10 ep, ES=4)",
    "total_params": total_p1, "train_sec": dt1, "best_val_f1": best_f1_1,
    "best_state": best_s1, "history": hist_e1, **res_e1,
}
print(f"--> Exp 01 Test Loss: {res_e1['test_loss']:.4f} | Test Acc: {res_e1['test_acc']*100:.2f}% | Test Macro-F1: {res_e1['test_macro_f1']:.4f}")
del model_e1, base_e1; free_gpu()

### Exp 02: Progressive Fine-Tuning (2 Tahap × 8 Epoch, EarlyStopping patience=3/tahap, Loader Sama dengan Exp 01)

In [ ]:
# Exp 02: Progressive Fine-Tuning (2 Tahap × 8 Epoch, EarlyStopping patience=3/tahap, Augmentasi SAMA dengan Exp 01)
seed_everything(SEED)
EPOCHS_E2_STAGE = 8
PATIENCE_E2_STAGE = get_early_stopping_patience(EPOCHS_E2_STAGE)  # 8 -> 3

base_e2 = build_model_for_exp("Exp02")
total_p2 = sum(p.numel() for p in base_e2.parameters())
criterion_e2 = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
hist_e2 = {"phase": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "train_f1": [], "val_f1": []}

t0 = time.time()

# Tahap 1 (8 Epoch, ES=3): Warm-up Classifier Head (Backbone Beku)
pg_s1 = set_freeze_exp01(base_e2)
model_e2 = wrap_multi_gpu(base_e2)
opt_s1 = torch.optim.AdamW(pg_s1, weight_decay=1e-4)
sch_s1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_s1, T_max=EPOCHS_E2_STAGE, eta_min=1e-5)
best_s2, best_f1_2 = train_phase(
    model_e2, train_loader_basic, val_loader, criterion_e2, opt_s1, sch_s1,
    epochs=EPOCHS_E2_STAGE, history=hist_e2, patience=PATIENCE_E2_STAGE, phase_name="Exp02-Stage1-FE"
)

# Tahap 2 (8 Epoch, ES=3): Unfreeze Blok Konvolusi Atas + Differential LR
unwrap_model(model_e2).load_state_dict(best_s2)
pg_s2 = set_unfreeze_exp02_stage2(base_e2)
trainable_s2 = sum(p.numel() for p in base_e2.parameters() if p.requires_grad)
print(f"[Exp 02 Stage 2] Trainable params: {trainable_s2:,} / {total_p2:,}")

opt_s2 = torch.optim.AdamW(pg_s2, weight_decay=1e-4)
sch_s2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_s2, T_max=EPOCHS_E2_STAGE, eta_min=1e-6)
best_s2, best_f1_2 = train_phase(
    model_e2, train_loader_basic, val_loader, criterion_e2, opt_s2, sch_s2,
    epochs=EPOCHS_E2_STAGE, history=hist_e2, patience=PATIENCE_E2_STAGE, phase_name="Exp02-Stage2-FT",
    best_state=best_s2, best_val_f1=best_f1_2
)

dt2 = time.time() - t0
res_e2 = evaluate_on_test(model_e2, best_s2, test_loader, criterion_e2)
ALL_EXPERIMENTS["Exp02"] = {
    "desc": "Progressive FT (8+8 ep, ES=3)",
    "total_params": total_p2, "train_sec": dt2, "best_val_f1": best_f1_2,
    "best_state": best_s2, "history": hist_e2, **res_e2,
}
print(f"--> Exp 02 Test Loss: {res_e2['test_loss']:.4f} | Test Acc: {res_e2['test_acc']*100:.2f}% | Test Macro-F1: {res_e2['test_macro_f1']:.4f}")
del model_e2, base_e2; free_gpu()

### Exp 03: Advanced Generalization (Maks. 20 Epoch, EarlyStopping patience=6, Full Unfreeze + Diff LR + Aggr Aug)

In [ ]:
# Exp 03: Advanced Generalization (Unfreeze SEMUA Layer + 3-Tier Differential LR + Augmentasi Agresif)
# Mulai dari bobot head terlatih (Exp 01) untuk mencegah catastrophic forgetting saat full unfreeze (LP-FT)
seed_everything(SEED)
EPOCHS_E3 = 20
PATIENCE_E3 = get_early_stopping_patience(EPOCHS_E3)  # 20 -> 6

base_e3 = build_model_for_exp("Exp03")
base_e3.load_state_dict(ALL_EXPERIMENTS["Exp01"]["best_state"])
total_p3 = sum(p.numel() for p in base_e3.parameters())
pg_e3 = set_full_unfreeze_exp03(base_e3)
trainable_e3 = sum(p.numel() for p in base_e3.parameters() if p.requires_grad)
print(f"[Exp 03 Full Unfreeze] Trainable params: {trainable_e3:,} / {total_p3:,}")
model_e3 = wrap_multi_gpu(base_e3)

criterion_e3 = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.08)
opt_e3 = torch.optim.AdamW(pg_e3, weight_decay=1e-2)
sch_e3 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_e3, T_max=EPOCHS_E3, eta_min=1e-6)

hist_e3 = {"phase": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "train_f1": [], "val_f1": []}
t0 = time.time()
best_s3, best_f1_3 = train_phase(
    model_e3, train_loader_aggr, val_loader, criterion_e3, opt_e3, sch_e3,
    epochs=EPOCHS_E3, history=hist_e3, patience=PATIENCE_E3, phase_name="Exp03-AdvGen", grad_clip=1.0
)
dt3 = time.time() - t0
res_e3 = evaluate_on_test(model_e3, best_s3, test_loader, criterion_e3)
ALL_EXPERIMENTS["Exp03"] = {
    "desc": "Full Unfreeze + Diff LR (20 ep, ES=6)",
    "total_params": total_p3, "train_sec": dt3, "best_val_f1": best_f1_3,
    "best_state": best_s3, "history": hist_e3, **res_e3,
}
print(f"--> Exp 03 Test Loss: {res_e3['test_loss']:.4f} | Test Acc: {res_e3['test_acc']*100:.2f}% | Test Macro-F1: {res_e3['test_macro_f1']:.4f}")
del model_e3, base_e3; free_gpu()

### Visualisasi Training & Validasi: Kurva Loss & Akurasi (Train vs Val vs Test) — Diagnosa Overfitting

In [ ]:
# ==============================================================================
# VISUALISASI TRAINING & VALIDASI: KURVA OVERFITTING (TRAIN vs VAL vs TEST)
# UNTUK MASING-MASING EKSPERIMEN (EXP 01, EXP 02, EXP 03)
# ==============================================================================
fig, axes = plt.subplots(3, 2, figsize=(14.5, 13.5))

for row_idx, exp_key in enumerate(["Exp01", "Exp02", "Exp03"]):
    r = ALL_EXPERIMENTS[exp_key]
    hist = r["history"]
    ep = np.arange(1, len(hist["train_loss"]) + 1)
    best_ep = int(np.argmax(hist["val_f1"])) + 1
    gap_acc = (hist["train_acc"][best_ep - 1] - hist["val_acc"][best_ep - 1]) * 100.0

    # Cek batas Stage 1 -> Stage 2 pada Exp 02
    stage_boundary = None
    if len(set(hist["phase"])) > 1:
        first_phase = hist["phase"][0]
        for idx_p, ph in enumerate(hist["phase"]):
            if ph != first_phase:
                stage_boundary = idx_p + 0.5
                break

    # --- Kolom Kiri: Kurva Loss (Train vs Val vs Garis Test Loss) ---
    ax_l = axes[row_idx, 0]
    ax_l.plot(ep, hist["train_loss"], label="Train Loss", color="#2563EB", marker="o", markersize=4, lw=2)
    ax_l.plot(ep, hist["val_loss"],   label="Val Loss",   color="#F59E0B", marker="s", markersize=4, lw=2)
    ax_l.axhline(r["test_loss"], color="#10B981", linestyle="--", lw=1.8, label=f"Test Loss ({r['test_loss']:.4f})")
    ax_l.axvline(best_ep, color="#EF4444", linestyle=":", lw=1.5, label=f"Best Ep ({best_ep})")
    if stage_boundary is not None:
        ax_l.axvline(stage_boundary, color="#6B7280", linestyle="-.", lw=1.3, label="Stage 1 → Stage 2")
    ax_l.set_title(f"{exp_key} — Kurva Loss ({r['desc']})", fontsize=10.5, fontweight="bold")
    ax_l.set_xlabel("Epoch")
    ax_l.set_ylabel("Cross-Entropy Loss")
    ax_l.legend(fontsize=8.5, loc="upper right")
    ax_l.grid(alpha=0.3)

    # --- Kolom Kanan: Kurva Akurasi (Train vs Val vs Garis Test Acc) ---
    ax_a = axes[row_idx, 1]
    ax_a.plot(ep, hist["train_acc"], label="Train Acc", color="#2563EB", marker="o", markersize=4, lw=2)
    ax_a.plot(ep, hist["val_acc"],   label="Val Acc",   color="#F59E0B", marker="s", markersize=4, lw=2)
    ax_a.axhline(r["test_acc"], color="#10B981", linestyle="--", lw=1.8, label=f"Test Acc ({r['test_acc']*100:.2f}%)")
    ax_a.axvline(best_ep, color="#EF4444", linestyle=":", lw=1.5, label=f"Best Ep ({best_ep})")
    if stage_boundary is not None:
        ax_a.axvline(stage_boundary, color="#6B7280", linestyle="-.", lw=1.3, label="Stage 1 → Stage 2")
    ax_a.set_ylim(max(0.0, min(hist["train_acc"] + hist["val_acc"]) - 0.08), 1.03)
    ax_a.set_title(
        f"{exp_key} — Kurva Akurasi | Gap(Tr-Va)@Best: {gap_acc:+.2f}%",
        fontsize=10.5, fontweight="bold",
    )
    ax_a.set_xlabel("Epoch")
    ax_a.set_ylabel("Accuracy")
    ax_a.legend(fontsize=8.5, loc="lower right")
    ax_a.grid(alpha=0.3)

plt.suptitle(f"Visualisasi Training & Diagnosa Overfitting (Train vs Val vs Test) — {MODEL_NAME}", fontsize=13.5, fontweight="bold", y=1.005)
plt.tight_layout()
plt.savefig(WORK_DIR / f"{MODEL_SLUG}_train_val_test_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Grafik Ringkasan Validasi Lintas 3 Eksperimen
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
colors = {"Exp01": "#64748B", "Exp02": "#F59E0B", "Exp03": "#10B981"}
for exp_key in ["Exp01", "Exp02", "Exp03"]:
    hist = ALL_EXPERIMENTS[exp_key]["history"]
    ep = range(1, len(hist["val_loss"]) + 1)
    axes[0].plot(ep, hist["val_loss"], marker="o", markersize=3.5, lw=2, label=exp_key, color=colors[exp_key])
    axes[1].plot(ep, hist["val_f1"],   marker="s", markersize=3.5, lw=2, label=exp_key, color=colors[exp_key])

axes[0].set_title(f"{MODEL_NAME} — Perbandingan Validation Loss (Exp01 vs Exp02 vs Exp03)")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Validation Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_title(f"{MODEL_NAME} — Perbandingan Validation Macro-F1 (Exp01 vs Exp02 vs Exp03)")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Validation Macro-F1"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(WORK_DIR / f"{MODEL_SLUG}_validation_comparison.png", dpi=150)
plt.show()

## 7. Testing (Visualisasi Prediksi Benar, Error Analysis, Confidence, & Grad-CAM)

In [ ]:
# ==============================================================================
# TESTING (VISUALISASI): PREDIKSI BENAR, ERROR ANALYSIS (SALAH), CONFIDENCE, & GRAD-CAM
# ==============================================================================
# Pilih eksperimen terbaik berdasarkan Best Val Macro-F1 (tanpa mengintip test set)
best_exp_key = max(["Exp01", "Exp02", "Exp03"], key=lambda k: ALL_EXPERIMENTS[k]["best_val_f1"])
best_exp = ALL_EXPERIMENTS[best_exp_key]
y_true = best_exp["y_true"]
y_pred = best_exp["y_pred"]
y_probs = best_exp["y_probs"]
confs = y_probs.max(axis=1)

print(f">>> MODEL TERBAIK TERPILIH ({MODEL_NAME}): {best_exp_key} — {best_exp['desc']}")
print(f"    Test Loss: {best_exp['test_loss']:.4f} | Test Acc: {best_exp['test_acc']*100:.2f}% | Test Macro-F1: {best_exp['test_macro_f1']:.4f}")

# 1. Visualisasi Sampel Prediksi Test Set (4 Prediksi Benar + 4 Kasus Salah / Error Analysis)
correct_idx = np.where(y_true == y_pred)[0]
wrong_idx   = np.where(y_true != y_pred)[0]
print(f"Total Test: {len(y_true)} | Benar: {len(correct_idx)} | Salah (Misclassified): {len(wrong_idx)}")

# Ambil 1 contoh benar per kelas (maks 4) & 4 contoh salah dengan confidence tertinggi (hardest errors)
sel_correct = []
for cls_id in range(NUM_CLASSES):
    cands = [i for i in correct_idx if y_true[i] == cls_id]
    if cands:
        sel_correct.append(cands[0])
sel_wrong = wrong_idx[np.argsort(-confs[wrong_idx])[:4]].tolist() if len(wrong_idx) > 0 else []

n_rows_vis = 2 if len(sel_wrong) > 0 else 1
fig, axes = plt.subplots(n_rows_vis, 4, figsize=(13.5, 3.4 * n_rows_vis))
axes = np.atleast_2d(axes)

for c in range(4):
    if c < len(sel_correct):
        idx = sel_correct[c]
        img_t, _ = test_ds[idx]
        axes[0, c].imshow(denorm(img_t))
        axes[0, c].set_title(
            f"[BENAR] True: {CLASS_NAMES[y_true[idx]]}\nPred: {CLASS_NAMES[y_pred[idx]]} ({confs[idx]*100:.1f}%)",
            fontsize=8.8, color="green", fontweight="bold"
        )
    axes[0, c].axis("off")

if n_rows_vis == 2:
    for c in range(4):
        if c < len(sel_wrong):
            idx = sel_wrong[c]
            img_t, _ = test_ds[idx]
            axes[1, c].imshow(denorm(img_t))
            axes[1, c].set_title(
                f"[SALAH] True: {CLASS_NAMES[y_true[idx]]}\nPred: {CLASS_NAMES[y_pred[idx]]} ({confs[idx]*100:.1f}%)",
                fontsize=8.8, color="red", fontweight="bold"
            )
        axes[1, c].axis("off")

plt.suptitle(f"Visualisasi Testing — Contoh Prediksi Benar vs Error Analysis ({MODEL_NAME} · {best_exp_key})", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(WORK_DIR / f"{MODEL_SLUG}_testing_predictions_errors.png", dpi=150)
plt.show()

# 2. Distribusi Confidence Score (Prediksi Benar vs Salah)
plt.figure(figsize=(8.5, 4.0))
plt.hist(confs[correct_idx], bins=20, range=(0.25, 1.0), alpha=0.75, label=f"Benar (n={len(correct_idx)})", color="#10B981")
if len(wrong_idx) > 0:
    plt.hist(confs[wrong_idx], bins=20, range=(0.25, 1.0), alpha=0.80, label=f"Salah (n={len(wrong_idx)})", color="#EF4444")
plt.title(f"Distribusi Confidence Score Prediksi Test Set — {MODEL_NAME} ({best_exp_key})")
plt.xlabel("Probabilitas Softmax Tertinggi (Confidence)")
plt.ylabel("Jumlah Citra Uji")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(WORK_DIR / f"{MODEL_SLUG}_confidence_histogram.png", dpi=150)
plt.show()

# 3. Visualisasi Explainability: Grad-CAM (Mengecek apakah model melihat lesi penyakit, bukan latar sawah)
best_model = build_model_for_exp(best_exp_key)
best_model.load_state_dict(best_exp["best_state"])
for _mod in best_model.modules():
    if hasattr(_mod, "inplace"):
        _mod.inplace = False
best_model.eval()

def compute_gradcam(model, target_layer, img_tensor, class_idx=None):
    activations = []
    def fwd_hook(module, inp, out):
        out.retain_grad()
        activations.append(out)

    handle = target_layer.register_forward_hook(fwd_hook)
    model.zero_grad()
    with torch.enable_grad():
        inp_req = img_tensor.clone().detach().requires_grad_(True)
        logits = model(inp_req)
        if class_idx is None:
            class_idx = int(logits.argmax(dim=1).item())
        logits[0, class_idx].backward()

    handle.remove()
    act = activations[0][0].detach()
    grad = activations[0].grad[0].detach()
    weights = grad.mean(dim=(1, 2), keepdim=True)
    cam = torch.relu((weights * act).sum(dim=0)).cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam, class_idx, float(torch.softmax(logits, dim=1)[0, class_idx].item())

sample_indices = []
for cls_id in range(NUM_CLASSES):
    for idx, (_, label) in enumerate(test_ds.samples):
        if label == cls_id:
            sample_indices.append(idx)
            break

target_layer = get_gradcam_layer(best_model, best_exp_key)
fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(10.5, 3.2 * NUM_CLASSES))
for row, ds_idx in enumerate(sample_indices):
    img_t, true_lbl = test_ds[ds_idx]
    cam, pred_lbl, conf = compute_gradcam(best_model, target_layer, img_t.unsqueeze(0).to(DEVICE))
    img_vis = denorm(img_t)
    cam_resized = np.asarray(Image.fromarray((cam * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)) / 255.0
    heatmap = plt.cm.jet(cam_resized)[..., :3]
    overlay = np.clip(0.58 * img_vis + 0.42 * heatmap, 0, 1)

    axes[row, 0].imshow(img_vis)
    axes[row, 0].set_title(f"Asli: {CLASS_NAMES[true_lbl]}", fontsize=9.5)
    axes[row, 1].imshow(cam_resized, cmap="jet")
    axes[row, 1].set_title("Grad-CAM Heatmap", fontsize=9.5)
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title(
        f"Pred: {CLASS_NAMES[pred_lbl]} ({conf*100:.1f}%)",
        fontsize=9.5,
        color=("green" if pred_lbl == true_lbl else "red"),
    )
    for c in range(3):
        axes[row, c].axis("off")

plt.suptitle(f"Visualisasi Grad-CAM — {MODEL_NAME} ({best_exp_key})", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(WORK_DIR / f"{MODEL_SLUG}_best_gradcam.png", dpi=150)
plt.show()

## 8. Evaluasi Model Komprehensif (Rekap Ablation, Confusion Matrix, Metrik per Kelas, & ROC-AUC)

In [ ]:
# ==============================================================================
# EVALUASI MODEL KOMPREHENSIF: TABEL ABLATION, CONFUSION MATRIX, METRIK PER KELAS,
# KURVA ROC-AUC MULTI-KELAS, SIMPAN CHECKPOINT & EKSPOR JSON
# ==============================================================================
rows = []
for exp_key in ["Exp01", "Exp02", "Exp03"]:
    r = ALL_EXPERIMENTS[exp_key]
    hist = r["history"]
    best_ep = int(np.argmax(hist["val_f1"]))
    ep_run = len(hist["val_f1"])
    es_info = hist.get("early_stopped_phases", [])
    es_flag = any(x["early_stopped"] for x in es_info)
    rows.append({
        "Eksperimen": exp_key,
        "Deskripsi": r["desc"],
        "Epochs Run": ep_run,
        "Early Stopped": es_flag,
        "Train Acc@Best": round(hist["train_acc"][best_ep], 4),
        "Val Acc@Best": round(hist["val_acc"][best_ep], 4),
        "Best Val F1": round(r["best_val_f1"], 4),
        "Test Loss": round(r["test_loss"], 4),
        "Test Acc": round(r["test_acc"], 4),
        "Test Macro-F1": round(r["test_macro_f1"], 4),
        "Waktu (s)": round(r["train_sec"], 1),
        "Inferensi (ms/img)": round(r["inference_ms"], 3),
    })

df_exp = pd.DataFrame(rows)
print(f"=== TABEL EVALUASI ABLATION STUDY: {MODEL_NAME} ===")
display(df_exp)

# Bar Chart Komparasi Test Accuracy & Macro-F1 (Exp 01 vs Exp 02 vs Exp 03)
fig, ax = plt.subplots(figsize=(8.5, 4.3))
x = np.arange(len(df_exp))
w = 0.34
ax.bar(x - w/2, df_exp["Test Acc"], width=w, label="Test Accuracy", color="#2563EB")
ax.bar(x + w/2, df_exp["Test Macro-F1"], width=w, label="Test Macro-F1", color="#0D9488")
ax.set_xticks(x)
ax.set_xticklabels(df_exp["Eksperimen"])
ax.set_ylim(0, 1.09)
ax.set_title(f"{MODEL_NAME} — Perbandingan Performa Test Set (Exp 01 vs Exp 02 vs Exp 03)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
for i, (a, f1) in enumerate(zip(df_exp["Test Acc"], df_exp["Test Macro-F1"])):
    ax.text(i - w/2, a + 0.015, f"{a*100:.2f}%", ha="center", fontsize=9, fontweight="bold")
    ax.text(i + w/2, f1 + 0.015, f"{f1:.4f}", ha="center", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.savefig(WORK_DIR / f"{MODEL_SLUG}_ablation_bar.png", dpi=150)
plt.show()

# Classification Report lengkap untuk Eksperimen Terbaik
print(f"\n=== CLASSIFICATION REPORT ({MODEL_NAME} · {best_exp_key}) ===")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

# Dual Confusion Matrix: Jumlah Absolut & Persentase Ternormalisasi (Recall per Kelas)
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title(f"Confusion Matrix (Jumlah) — {best_exp_key}\nTest Acc: {best_exp['test_acc']*100:.2f}%")
axes[0].set_xlabel("Prediksi Model"); axes[0].set_ylabel("Label Aktual")

sns.heatmap(cm_norm * 100, annot=True, fmt=".1f", cmap="YlGnBu", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title(f"Confusion Matrix Ternormalisasi (%) — {best_exp_key}")
axes[1].set_xlabel("Prediksi Model"); axes[1].set_ylabel("Label Aktual")
plt.tight_layout()
plt.savefig(WORK_DIR / f"{MODEL_SLUG}_best_confusion_matrix.png", dpi=150)
plt.show()

# Bar Chart Precision, Recall, F1-Score per Kelas + Kurva ROC-AUC Multi-Kelas (One-vs-Rest)
prec, rec, f1c, _ = precision_recall_fscore_support(y_true, y_pred, average=None)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

xc = np.arange(NUM_CLASSES)
wc = 0.25
axes[0].bar(xc - wc, prec, width=wc, label="Precision", color="#3B82F6")
axes[0].bar(xc,      rec,  width=wc, label="Recall",    color="#F59E0B")
axes[0].bar(xc + wc, f1c,  width=wc, label="F1-Score",  color="#10B981")
axes[0].set_xticks(xc)
axes[0].set_xticklabels(CLASS_NAMES, rotation=15)
axes[0].set_ylim(0, 1.09)
axes[0].set_title(f"Metrik Evaluasi per Kelas — {MODEL_NAME} ({best_exp_key})")
axes[0].set_ylabel("Skor")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
roc_aucs = {}
roc_colors = ["#2563EB", "#EF4444", "#F59E0B", "#10B981"]
for i, cls in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
    roc_auc = auc(fpr, tpr)
    roc_aucs[cls] = round(float(roc_auc), 4)
    axes[1].plot(fpr, tpr, lw=2, color=roc_colors[i % len(roc_colors)], label=f"{cls} (AUC = {roc_auc:.4f})")
axes[1].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.6)
axes[1].set_title(f"Kurva ROC-AUC Multi-Kelas (OvR) — {MODEL_NAME} ({best_exp_key})")
axes[1].set_xlabel("False Positive Rate (FPR)")
axes[1].set_ylabel("True Positive Rate (TPR)")
axes[1].legend(loc="lower right", fontsize=9)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(WORK_DIR / f"{MODEL_SLUG}_per_class_and_roc_auc.png", dpi=150)
plt.show()

# Simpan bobot terbaik & rekap lengkap JSON untuk komparasi antar 4 arsitektur
ckpt_path = WORK_DIR / f"{MODEL_SLUG}_{best_exp_key.lower()}_best.pth"
torch.save(best_exp["best_state"], ckpt_path)
model_size_mb = ckpt_path.stat().st_size / (1024 * 1024)
print(f"Checkpoint tersimpan: {ckpt_path} ({model_size_mb:.2f} MB)")

export_json = {
    "model_name": MODEL_NAME,
    "model_slug": MODEL_SLUG,
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "num_gpus": NUM_GPUS,
    "multi_gpu_dataparallel": USE_MULTI_GPU,
    "best_experiment": best_exp_key,
    "best_model_size_mb": round(model_size_mb, 2),
    "best_roc_auc_per_class": roc_aucs,
    "experiments": {
        k: {
            "desc": ALL_EXPERIMENTS[k]["desc"],
            "total_params": int(ALL_EXPERIMENTS[k]["total_params"]),
            "epochs_run": len(ALL_EXPERIMENTS[k]["history"]["val_f1"]),
            "early_stopping_log": ALL_EXPERIMENTS[k]["history"].get("early_stopped_phases", []),
            "train_time_sec": round(ALL_EXPERIMENTS[k]["train_sec"], 2),
            "best_val_macro_f1": round(ALL_EXPERIMENTS[k]["best_val_f1"], 4),
            "test_loss": round(ALL_EXPERIMENTS[k]["test_loss"], 4),
            "test_accuracy": round(ALL_EXPERIMENTS[k]["test_acc"], 4),
            "test_macro_f1": round(ALL_EXPERIMENTS[k]["test_macro_f1"], 4),
            "inference_ms_per_image": round(ALL_EXPERIMENTS[k]["inference_ms"], 3),
        }
        for k in ["Exp01", "Exp02", "Exp03"]
    },
}
json_out = WORK_DIR / f"{MODEL_SLUG}_results.json"
with open(json_out, "w") as f:
    json.dump(export_json, f, indent=2)
print("Rekap JSON tersimpan di:", json_out)